# S5+classification

한 슬라이스는 인코더를 지난 뒤 두 갈래로 나뉜다. 디코더는 μ와 B0–B3 skip으로 마스크를 그린다. 헤드는 같은 μ로 병변 존재 확률을 낸다. 그 확률이 검증에서 고른 τ보다 낮으면 마스크를 비운다.

1. `Encoder`가 μ, logvar, z, skip(B0–B4)를 낸다.
2. `Decoder`는 μ와 B0–B3 skip으로 마스크 로짓을 만든다. B4는 쓰지 않는다.
3. 학습이 끝난 분할기는 고정한다. `LesionPresenceHead`만 μ에서 로짓을 낸다. 시그모이드는 헤드 안에 없다.
4. 손실은 `BCEWithLogits`이다. 양성 가중치는 2이다.
5. 확률은 `p = sigmoid(로짓)`이다.
6. 검증 Dice가 최대인 τ로 테스트 마스크를 게이트한다.

S5 가중치는 `results/covid/label_ratio/rXX/s5/seg_best.pt`에 있어야 한다. 이 노트북은 그 파일을 다시 학습하지 않는다.

# S5+classification

After the encoder, one slice splits into two branches. The decoder draws the mask from μ and the B0–B3 skips. The head reads the same μ and estimates the probability that a lesion is present. When that probability is below the τ chosen on the validation set, the mask is cleared.

1. `Encoder` returns μ, logvar, z, and skips B0–B4.
2. `Decoder` builds the mask logit from μ and skips B0–B3. B4 is unused.
3. The trained segmenter is frozen. Only `LesionPresenceHead` reads μ and emits a logit. There is no sigmoid inside the head.
4. The loss is `BCEWithLogits`. The positive weight is 2.
5. The probability is `p = sigmoid(logit)`.
6. Test masks are gated by the τ that maximizes validation Dice.

The S5 weights must already be at `results/covid/label_ratio/rXX/s5/seg_best.pt`. This notebook does not train that file again.

In [ ]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from pipeline.config import RESULT_DIR
from pipeline.engine import DEVICE, atomic_torch_save, batch_dice, batch_pred_gt_sums, safe_load
from pipeline.metrics import binary_auroc, metrics_bundle
from pipeline.stage_segment import make_bundle

RATIO = 0.3
CLS_EPOCHS = 40
CLS_ES = 10
CLS_POS_WEIGHT = 2.0
RUN = False  # True로 바꾸면 헤드를 학습한다. report.json이 있으면 건너뛴다.

## 인코더와 S5 디코더

`base=32`이면 병목 채널은 1024이다. 인코더는 해상도를 다섯 줄이며 B0–B4 skip을 남긴다. 디코더는 마스크를 만들 때 B0–B3만 concat한 뒤 3×3으로 합친다. 분류 헤드는 이 skip을 보지 않고 μ만 받는다.

## Encoder and S5 decoder

With `base=32` the bottleneck has 1024 channels. The encoder downsamples five times and keeps skips B0–B4. When drawing the mask, the decoder concatenates only B0–B3 and fuses each pair with a 3×3 convolution. The classification head does not see these skips. It receives only μ.

In [ ]:
class ResBlockUp(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_out),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)
        return conv2 + conv3


class ResBlockDown(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        if self.act:
            conv2 = self.lrelu(conv2)
        conv3 = self.conv3_block(x)
        if self.act:
            conv3 = self.lrelu(conv3)
        return conv2 + conv3


class ConcatConvFuse(nn.Module):
    """cat(decoder, skip) -> 3x3 Conv -> BN -> LeakyReLU."""

    def __init__(self, dec_ch: int, skip_ch: int):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv2d(dec_ch + skip_ch, dec_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        if skip.shape[-2:] != x.shape[-2:]:
            skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([x, skip], dim=1))


class Encoder(nn.Module):
    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder_initial_flair = self._create_initial_encoder(1, base, dropout_rate)
        self.encoder_features_flair = self._create_feature_encoder(base, dropout_rate)
        self.encoder_mu_flair = self._create_mu_encoder(32 * base)
        self.encoder_logvar_flair = self._create_logvar_encoder(32 * base)

    def _create_initial_encoder(self, in_channels, base, dropout_rate):
        return nn.Sequential(
            nn.Conv2d(in_channels, base, 3, stride=1, padding=1),
            nn.BatchNorm2d(base),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_feature_encoder(self, base, dropout_rate):
        return nn.Sequential(
            ResBlockDown(base, 2 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(2 * base, 4 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(4 * base, 8 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(8 * base, 16 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(16 * base, 32 * base),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_mu_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def _create_logvar_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward_features(self, flair):
        e0 = self.encoder_initial_flair(flair)
        e1 = self.encoder_features_flair[0](e0)
        e1d = self.encoder_features_flair[1](e1)
        e2 = self.encoder_features_flair[2](e1d)
        e2d = self.encoder_features_flair[3](e2)
        e3 = self.encoder_features_flair[4](e2d)
        e3d = self.encoder_features_flair[5](e3)
        e4 = self.encoder_features_flair[6](e3d)
        e4d = self.encoder_features_flair[7](e4)
        h = self.encoder_features_flair[8](e4d)
        h = self.encoder_features_flair[9](h)
        mu = self.encoder_mu_flair(h)
        logvar = self.encoder_logvar_flair(h)
        z = self.reparameterize(mu, logvar)
        return mu, logvar, z, (e0, e1, e2, e3, e4)

    def forward(self, flair):
        mu, logvar, z, _ = self.forward_features(flair)
        return mu, logvar, z


class Decoder(nn.Module):
    """S5: B0..B3 concat+3x3 fuse, B4 off."""

    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        feat = 32 * base
        self.up1 = ResBlockUp(feat, feat)
        self.drop1 = nn.Dropout2d(p=dropout_rate)
        self.up2 = ResBlockUp(feat, feat)
        self.drop2 = nn.Dropout2d(p=dropout_rate)
        self.up3 = ResBlockUp(feat, 16 * base)
        self.drop3 = nn.Dropout2d(p=dropout_rate)
        self.up4 = ResBlockUp(16 * base, 8 * base)
        self.drop4 = nn.Dropout2d(p=dropout_rate)
        self.up5 = ResBlockUp(8 * base, base)
        self.drop5 = nn.Dropout2d(p=dropout_rate)
        self.out_conv = nn.Conv2d(base, 1, 3, stride=1, padding=1)

        self.fuse3 = ConcatConvFuse(feat, 8 * base)
        self.fuse2 = ConcatConvFuse(16 * base, 4 * base)
        self.fuse1 = ConcatConvFuse(8 * base, 2 * base)
        self.fuse0 = ConcatConvFuse(base, base)

    def forward(self, z, skips):
        e0, e1, e2, e3, e4 = skips
        del e4

        x = self.drop1(self.up1(z))
        x = self.drop2(self.up2(x))
        x = self.fuse3(x, e3)
        x = self.drop3(self.up3(x))
        x = self.fuse2(x, e2)
        x = self.drop4(self.up4(x))
        x = self.fuse1(x, e1)
        x = self.drop5(self.up5(x))
        x = self.fuse0(x, e0)
        return self.out_conv(x)


class VAE(nn.Module):
    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(base, dropout_rate)
        self.decoder = Decoder(base, dropout_rate)

    def forward(self, x):
        mu, logvar, z, skips = self.encoder.forward_features(x)
        out = self.decoder(z, skips)
        return out, mu, logvar


## 헤드

μ는 `1024 = 32×32` 채널의 공간 특징이다. 평균 풀링과 최대 풀링을 이어 붙여 로짓 하나를 낸다.

## Head

μ is a spatial map with 1024 = 32×32 channels. Average pooling and max pooling are concatenated and mapped to a single logit.

In [ ]:
class LesionPresenceHead(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, feature: torch.Tensor) -> torch.Tensor:
        pooled = torch.cat([
            self.avg_pool(feature).flatten(1),
            self.max_pool(feature).flatten(1),
        ], dim=1)
        return self.classifier(pooled).squeeze(1)

## 확률과 τ

마스크 로짓은 S5 디코더가 만든다. 존재 확률은 같은 μ에서 따로 만든다. τ는 검증 집합에서 전체 평균 Dice가 가장 높은 값이다.

## Probability and τ

The S5 decoder produces the mask logit. The presence probability is computed separately from the same μ. τ is the threshold that maximizes mean Dice on the validation set.

In [ ]:
def prep_xy(batch):
    x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
    if x.ndim == 3:
        x = x.unsqueeze(1)
    if y.ndim == 3:
        y = y.unsqueeze(1)
    return x, y


@torch.inference_mode()
def collect(model, head, loader):
    model.eval()
    head.eval()
    bags = {"gt": [], "mass": [], "dice": [], "score": []}
    for batch in loader:
        x, y = prep_xy(batch)
        mu, _lv, _z, skips = model.encoder.forward_features(x)
        seg_logits = model.decoder(mu, skips)
        mass, gt = batch_pred_gt_sums(seg_logits, y)
        bags["gt"].append(gt)
        bags["mass"].append(mass)
        bags["dice"].append(batch_dice(seg_logits, y))
        bags["score"].append(torch.sigmoid(head(mu)).detach().cpu().numpy())
    return {k: np.concatenate(v) for k, v in bags.items()}


def pick_tau(val):
    best = None
    for i in range(1, 100):
        thr = i / 100.0
        keep = val["score"] >= thr
        m = metrics_bundle(val["gt"], val["mass"], val["dice"], keep)
        key = (m["overall_dice"], -m["empty_fp_rate"])
        if best is None or key > best["key"]:
            best = {"thr": thr, "key": key, "m": m}
    return best["thr"], best["m"]

## 학습

타깃은 슬라이스에 병변 픽셀이 있으면 1, 없으면 0이다. 손실에 넣는 값은 확률이 아니라 로짓이다. 검증 AUROC가 10 에폭 동안 오르지 않으면 멈춘다.

## Training

The target is 1 when the slice contains any lesion pixel, and 0 otherwise. The loss takes the logit, not the probability. Training stops when the validation AUROC does not rise for 10 epochs.

In [ ]:
def train_s5_cls(ratio=RATIO):
    pct = int(round(ratio * 100))
    seg_ckpt = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5" / "seg_best.pt"
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5_cls"
    out_dir.mkdir(parents=True, exist_ok=True)
    report_path = out_dir / "report.json"
    if report_path.exists():
        print("skip", report_path)
        return json.loads(report_path.read_text(encoding="utf-8"))

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, 4)
    model = VAE(base=32).to(DEVICE)
    model.load_state_dict(safe_load(seg_ckpt, map_location="cpu")["model"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = LesionPresenceHead(32 * 32).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CLS_POS_WEIGHT], device=DEVICE))
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auroc, bad, best_ep = -1.0, 0, 0
    best_path = out_dir / "head_best.pt"
    trend = []

    for ep in range(1, CLS_EPOCHS + 1):
        head.train()
        total, n = 0.0, 0
        for batch in train_loader:
            x, y = prep_xy(batch)
            with torch.no_grad():
                mu, _lv, _z, _sk = model.encoder.forward_features(x)
            y_cls = (y.flatten(1).sum(1) > 0).float()
            opt.zero_grad(set_to_none=True)
            loss = bce(head(mu), y_cls)
            loss.backward()
            opt.step()
            total += float(loss.item()) * y_cls.size(0)
            n += int(y_cls.size(0))
        val = collect(model, head, val_loader)
        auroc = binary_auroc(val["gt"] > 0, val["score"])
        trend.append({"ep": ep, "train_bce": total / max(n, 1), "val_auroc": auroc})
        print(f"ep {ep} bce={trend[-1]['train_bce']:.4f} auroc={auroc:.4f}")
        if auroc > best_auroc + 1e-4:
            best_auroc, bad, best_ep = auroc, 0, ep
            atomic_torch_save({"head": head.state_dict(), "ep": ep, "auroc": auroc}, best_path)
        else:
            bad += 1
            if bad >= CLS_ES:
                break

    head.load_state_dict(safe_load(best_path, map_location="cpu")["head"])
    val = collect(model, head, val_loader)
    test = collect(model, head, test_loader)
    tau, _val_m = pick_tau(val)
    mt = metrics_bundle(test["gt"], test["mass"], test["dice"], test["score"] >= tau)
    report = {
        "ratio": ratio,
        "n_train": n_train,
        "loss": "BCEWithLogits",
        "probability": "p = sigmoid(Head(mu)) = P(lesion present | slice)",
        "tau": tau,
        "best_ep": best_ep,
        "val_auroc": best_auroc,
        "test_dice": float(mt["overall_dice"]),
        "test_nonempty_dice": float(mt["nonempty_dice"]),
        "test_empty_fp_rate": float(mt["empty_fp_rate"]),
    }
    (out_dir / "prob_trend.json").write_text(json.dumps(trend, indent=2), encoding="utf-8")
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return report


if RUN:
    train_s5_cls(RATIO)